In [24]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

#####################
      ## OLS ##
#####################

def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs

ind_regs = OLS(ind_spf_trim, '2016-12-31')
%store ind_regs
ind_regs[3].summary()

Stored 'ind_regs' (list)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     20.71
Date:                Mon, 22 Apr 2024   Prob (F-statistic):           5.54e-06
Time:                        08:55:15   Log-Likelihood:                 10233.
No. Observations:                3424   AIC:                        -2.046e+04
Df Residuals:                    3422   BIC:                        -2.045e+04
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0023      0.000     -8.002      0.000      -0.003      -0.002
r_t3          -0.2021      0.044     -4.550      0.000      -0.289      -0.115
==============================================================================
Omnibus:                      116.815   Durbin-Watson:                   0.406
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              219.697
Skew:                          -0.257   Prob(JB):                     1.97e-48
Kurtosis:                       4.129   Cond. No.                         140.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [25]:
mean_regs = OLS(mean_spf_trim, '2016-12-31')
%store mean_regs
mean_regs[3].summary()

Stored 'mean_regs' (list)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.005
Method:                 Least Squares   F-statistic:                     1.532
Date:                Mon, 22 Apr 2024   Prob (F-statistic):              0.218
Time:                        08:55:15   Log-Likelihood:                 440.50
No. Observations:                 141   AIC:                            -877.0
Df Residuals:                     139   BIC:                            -871.1
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0022      0.001     -1.750      0.082      -0.005       0.000
r_t3           0.2573      0.208      1.238      0.218      -0.154       0.668
==============================================================================
Omnibus:                       10.826   Durbin-Watson:                   0.640
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               15.671
Skew:                          -0.412   Prob(JB):                     0.000395
Kurtosis:                       4.410   Cond. No.                         219.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using 1 lags and without small sample correction
"""

In [26]:
#####################
     ## ID FE ##
#####################
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs

ind_regs_fe = ID_FE(ind_spf_trim, '2016-12-31')
%store ind_regs_fe
ind_regs_fe[3].summary()

Stored 'ind_regs_fe' (list)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.151
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     5.555
Date:                Mon, 22 Apr 2024   Prob (F-statistic):           7.36e-72
Time:                        08:55:16   Log-Likelihood:                 10490.
No. Observations:                3424   AIC:                        -2.072e+04
Df Residuals:                    3295   BIC:                        -1.993e+04
Df Model:                         128                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0155      0.004     -4.194      0.000      -0.023      -0.008
x1            -0.2718      0.043     -6.380      0.000      -0.355      -0.188
x2             0.0148      0.005      2.874      0.004       0.005       0.025
x3             0.0027      0.005      0.505      0.614      -0.008       0.013
x4             0.0132      0.005      2.821      0.005       0.004       0.022
x5             0.0030      0.004      0.833      0.405      -0.004       0.010
x6             0.0128      0.004      3.197      0.001       0.005       0.021
x7            -0.0050      0.006     -0.809      0.418      -0.017       0.007
x8             0.0116      0.004      2.611      0.009       0.003       0.020
x9            -0.0069      0.005     -1.363      0.173      -0.017       0.003
x10            0.0079      0.004      1.846      0.065      -0.000       0.016
x11            0.0096      0.004      2.467      0.014       0.002       0.017
x12            0.0122      0.008      1.488      0.137      -0.004       0.028
x13            0.0079      0.005      1.739      0.082      -0.001       0.017
x14            0.0179      0.006      3.249      0.001       0.007       0.029
x15            0.0104      0.008      1.271      0.204      -0.006       0.026
x16            0.0112      0.006      1.802      0.072      -0.001       0.023
x17            0.0095      0.004      2.324      0.020       0.001       0.017
x18            0.0094      0.004      2.262      0.024       0.001       0.018
x19            0.0146      0.004      3.773      0.000       0.007       0.022
x20            0.0056      0.004      1.260      0.208      -0.003       0.014
x21            0.0046      0.006      0.794      0.427      -0.007       0.016
x22            0.0018      0.005      0.388      0.698      -0.007       0.011
x23            0.0011      0.006      0.181      0.857      -0.011       0.013
x24            0.0094      0.004      2.141      0.032       0.001       0.018
x25            0.0130      0.006      2.247      0.025       0.002       0.024
x26           -0.0119      0.005     -2.599      0.009      -0.021      -0.003
x27            0.0058      0.004      1.308      0.191      -0.003       0.014
x28            0.0030      0.006      0.505      0.613      -0.009       0.015
x29            0.0030      0.004      0.704      0.482      -0.005       0.011
x30            0.0106      0.004      2.753      0.006       0.003       0.018
x31            0.0093      0.011      0.823      0.411      -0.013       0.032
x32            0.0115      0.004      2.904      0.004       0.004       0.019
x33            0.0140      0.004      3.396      0.001       0.006       0.022
x34            0.0137      0.004      3.091      0.002       0.005       0.022
x35            0.0090      0.004      2.374      0.018       0.002       0.016
x3

In [27]:
### Two-way Fixed Effects ###
def IDT_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float), pd.get_dummies(df['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=1))
    return regs

ind_regs_fe2 = IDT_FE(ind_spf_trim, '2016-12-31')
%store ind_regs_fe2
ind_regs_fe2[3].summary()

Stored 'ind_regs_fe2' (list)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.876
Method:                 Least Squares   F-statistic:                     165.2
Date:                Mon, 22 Apr 2024   Prob (F-statistic):               0.00
Time:                        08:55:18   Log-Likelihood:                 13929.
No. Observations:                3424   AIC:                        -2.732e+04
Df Residuals:                    3155   BIC:                        -2.567e+04
Df Model:                         268                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0238      0.002     -9.580      0.000      -0.029      -0.019
x1            -0.4744      0.027    -17.275      0.000      -0.528      -0.421
x2            -0.0006      0.002     -0.267      0.789      -0.005       0.004
x3             0.0016      0.002      0.759      0.448      -0.002       0.006
x4             0.0011      0.001      0.732      0.464      -0.002       0.004
x5             0.0024      0.001      1.738      0.082      -0.000       0.005
x6             0.0018      0.002      1.111      0.267      -0.001       0.005
x7            -0.0062      0.003     -2.028      0.043      -0.012      -0.000
x8             0.0031      0.002      1.270      0.204      -0.002       0.008
x9             0.0003      0.005      0.060      0.952      -0.009       0.009
x10            0.0006      0.002      0.397      0.691      -0.002       0.004
x11            0.0004      0.001      0.305      0.761      -0.002       0.003
x12            0.0111      0.005      2.094      0.036       0.001       0.021
x13            0.0036      0.001      2.376      0.018       0.001       0.006
x14            0.0038      0.002      2.305      0.021       0.001       0.007
x15            0.0088      0.007      1.336      0.182      -0.004       0.022
x16            0.0057      0.002      2.345      0.019       0.001       0.010
x17            0.0015      0.001      1.157      0.248      -0.001       0.004
x18            0.0023      0.002      1.364      0.173      -0.001       0.006
x19            0.0062      0.001      4.193      0.000       0.003       0.009
x20           -0.0006      0.002     -0.358      0.721      -0.004       0.003
x21            0.0033      0.002      1.424      0.154      -0.001       0.008
x22            0.0009      0.001      0.652      0.514      -0.002       0.004
x23           -0.0001      0.002     -0.063      0.950      -0.003       0.003
x24            0.0031      0.002      1.949      0.051   -1.93e-05       0.006
x25            0.0065      0.002      3.093      0.002       0.002       0.011
x26           -0.0009      0.003     -0.362      0.717      -0.006       0.004
x27            0.0021      0.001      1.622      0.105      -0.000       0.005
x28            0.0027      0.002      1.432      0.152      -0.001       0.006
x29           -0.0046      0.002     -3.015      0.003      -0.008      -0.002
x30            0.0006      0.001      0.405      0.686      -0.002       0.003
x31            0.0017      0.004      0.415      0.678      -0.006       0.010
x32            0.0038      0.001      2.759      0.006       0.001       0.007
x33            0.0060      0.002      2.788      0.005       0.002       0.010
x34           -0.0032      0.002     -1.429      0.153      -0.008       0.001
x35            0.0037      0.001      2.562      0.010       0.001       0.007
x3

In [28]:
#####################
    ## AR(1) ##
#####################

def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg

ar_1 = AR(vintage_trim, '2016-12-31')
%store ar_1
ar_1.summary()

Stored 'ar_1' (AutoRegResultsWrapper)


/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            AutoReg Model Results                             
==============================================================================
Dep. Variable:                     t3   No. Observations:                  207
Model:                     AutoReg(1)   Log Likelihood                 714.958
Method:               Conditional MLE   S.D. of innovations              0.008
Date:                Mon, 22 Apr 2024   AIC                          -1423.916
Time:                        08:55:19   BIC                          -1413.933
Sample:                             1   HQIC                         -1419.879
                                  207                                         
==============================================================================
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0014      0.001      1.514      0.130      -0.000       0.003
t3.L1          0.9657      0.018     53.046      0.000       0.930       1.001
                                    Roots                                    
=============================================================================
                  Real          Imaginary           Modulus         Frequency
-----------------------------------------------------------------------------
AR.1            1.0355           +0.0000j            1.0355            0.0000
-----------------------------------------------------------------------------
"""

In [29]:
###############################################
            ## MODEL PARAMETERS ##
###############################################

def params(ols, pldols, fe, fe2, ar1):
    params = pd.DataFrame(index=['ols', 'pld', 'fe', 'fe2'])
    params.loc['ols', 'lambda'] = ols / (1 + ols)
    params.loc['ols', 'G'] = 1 / (1 + ols)
    params.loc['pld', 'Theta'] = (-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1))
    params.loc['fe', 'Theta'] = (-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1))
    params.loc['fe2', 'Theta'] = (-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1))
    return params

parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ind_regs_fe2[3].params[1], ar_1.roots)
%store parameters
print(parameters)

Stored 'parameters' (DataFrame)
       lambda         G     Theta
ols  0.204613  0.795387       NaN
pld       NaN       NaN  0.268660
fe        NaN       NaN  0.423848
fe2       NaN       NaN  3.982421
